# Creating Parameter for Flag Purpose



In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import * 

In [0]:
   dbutils.widgets.text("Incremental_flag","0")
   

# Now we are checking the value which we assign to Incremental flag


In [0]:
incremental_flag = dbutils.widgets.get("Incremental_flag")
print(type(incremental_flag))

#Creating Dim Model


### Fetch Relative Columns

In [0]:
df_source = spark.sql("""
select Distinct(Date_ID) as Date_ID  from parquet.`abfss://silver-layer@adlcarproject.dfs.core.windows.net/carsalesdata`""").sort("Date_ID",ascending =True)

df_source.display()

### Dim_Model sink --- Inital and Incremental ( It will bring just schema if table will not exists)

In [0]:
if spark.catalog.tableExists("car_project.gold.dim_Date"):
    df_sink = spark.sql('''select 1 as dim_date_key, Date_id from parquet.`abfss://silver-layer@adlcarproject.dfs.core.windows.net/carsalesdata`
    where 1 = 0''')
else:
    df_sink = spark.sql('''
    select 1 as dim_date_key,date_id from parquet.`abfss://silver-layer@adlcarproject.dfs.core.windows.net/carsalesdata`
    where 1 = 0 ''')

# Filtering new record and old records (upsert)

In [0]:
df_filter = df_source.join(df_sink,df_source["date_id"] == df_sink["date_id"],"left").select(df_source["date_ID"],df_sink["dim_date_key"])

df_filter.display()

##Filter_old_records On basic of null


In [0]:
df_filter_old = df_filter.filter(col("dim_date_key").isNotNull())

In [0]:
df_filter_old.display()

## Bring all new records

In [0]:
df_filter_new = df_filter.filter(col("dim_date_key").isNull())
df_filter_new.display()

#Creating Surrogate Key


In [0]:
if "incremental_flag == 0":
    max_value = 1
else:
    max_value_df = spark.sql("select max(dim_date_key) from car_project.gold.dim_date")
    max_value  = max_value_df.collect()[0][0]

creating surrogate key column and adding the max surrogate key

In [0]:
df_filter_new= df_filter_new.withColumn("dim_date_key",max_value+monotonically_increasing_id())
df_filter_new.display()

# creating final_df --- To union of old and new df

In [0]:
df_final = df_filter_old.union(df_filter_new)
df_final.display()

# SCD Type--1 (Upsert)

In [0]:

from delta.tables import DeltaTable

In [0]:
# Incremental RUN
if spark.catalog.tableExists("car_project.gold.dim_date"):
    delta_table = DeltaTable.forPath(
        spark,
        "abfss://gold-layer@adlcarproject.dfs.core.windows.net/dim_date"
    )
    delta_table.alias("target").merge(
        df_final.alias("source"),
        "target.dim_date_key = source.dim_date_key"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
else:
    df_final.write.format("delta") \
        .mode("append") \
        .option("path", "abfss://gold-layer@adlcarproject.dfs.core.windows.net/dim_date") \
        .saveAsTable("car_project.gold.dim_date")

In [0]:
%sql
select * from car_project.gold.dim_date